In [1]:
!pip install openai


^C


In [5]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [6]:
tweets = pd.read_parquet(r'C:\Users\24558\Desktop\chapter-2\data\immigration_classified_tweets_german_parl.parquet')


In [7]:
# filter tweets where predicted == 1
tweets = tweets[tweets['predicted'] == 1]

In [8]:
tweets

,user_username,text,created_at,tweet_id,author_id,party,source_type,year_month,predicted
4,Renner_AfD,Deutschland rettet... und importiert dabei das...,2018-07-16 12:54:34+00:00,1018841321623183360,2535401947,AFD,twitter_parliamentarian,2018-07-15 00:00:00+00:00,1
5,Renner_AfD,Integration setzt Identifikation voraus. Ident...,2018-07-15 23:05:45+00:00,1018632741594779648,2535401947,AFD,twitter_parliamentarian,2018-07-15 00:00:00+00:00,1
9,Renner_AfD,"Es gibt Worte, die hasse ich wie die Pest, z. ...",2018-07-01 20:02:38+00:00,1013513229375557632,2535401947,AFD,twitter_parliamentarian,2018-07-01 00:00:00+00:00,1
10,Renner_AfD,Meine Bundestagsrede vom 28.06.2018 zum Thema ...,2018-06-29 13:12:26+00:00,1012685223132352513,2535401947,AFD,twitter_parliamentarian,2018-06-24 00:00:00+00:00,1
12,Renner_AfD,"Rechtsruck durch uns, die AfD? Nein! Die notwe...",2018-06-23 00:41:25+00:00,1010321894761992193,2535401947,AFD,twitter_parliamentarian,2018-06-17 00:00:00+00:00,1
...,...,...,...,...,...,...,...,...,...
1060633,Katrin_Werner,Am Samstag wurde eine Frau aus Hamburg in die ...,2018-08-07 12:29:28+00:00,1026807538132688896,224630872,Linke,twitter_parliamentarian,2018-08-05 00:00:00+00:00,1
1060641,Katrin_Werner,Statt die drängenden Probleme wie Pflegenotsta...,2018-07-09 06:51:15+00:00,1016213172594905088,224630872,Linke,twitter_parliamentarian,2018-07-08 00:00:00+00:00,1
1060649,Katrin_Werner,Über 68 Mio. Menschen sind auf der Flucht. Sie...,2018-06-20 07:21:25+00:00,1009335395002257408,224630872,Linke,twitter_parliamentarian,2018-06-17 00:00:00+00:00,1
1060653,Katrin_Werner,Wir LINKEN fordern: Konsequente Bekämpfung von...,2018-06-08 15:08:24+00:00,1005104260923707392,224630872,Linke,twitter_parliamentarian,2018-06-03 00:00:00+00:00,1


In [9]:
import re

def remove_urls(text):
    return re.sub(r'http[s]?://\S+', '', text)

# Apply the function to the 'text' column in your DataFrame
tweets['text'] = tweets['text'].apply(remove_urls)

In [10]:
def remove_hashtags(text):
    return re.sub(r'#', '', text)

# Apply the function to the 'text' column in your DataFrame
tweets['text'] = tweets['text'].apply(remove_hashtags)

In [ ]:
from openai import OpenAI
client = OpenAI(api_key="sk-your-api-key-here")
openai.api_key = "your api key again"

NameError: name 'openai' is not defined

In [ ]:
import openai
import time

# OpenAI Rate Limits (for gpt-4o-mini)
MAX_REQUESTS_PER_MIN = 60  # Adjust based on limits
REQUEST_INTERVAL = 60 / MAX_REQUESTS_PER_MIN  # Time interval per request (~1s)

# Global tracker for last request time
last_request_time = time.time()

def classify_tweet(tweet, max_retries=5):
    """Classifies a tweet using OpenAI, handling rate limits and retries."""
    global last_request_time

    prompt = f"""
    Please classify the following tweet based on its stance toward immigration, integration, refugees, or asylum.
    Be cautious about both direct and indirect quotes, and ensure that the speaker's stance is correctly identified.
    Use the number 1 for a positive stance,
    Use the number 2 for a negative stance,
    Use the number 0 for a neutral stance.

    Only return the numbers, never return any text or description.

    Tweet: {tweet}
    Classification:
    """

    retries = 0
    while retries < max_retries:
        try:
            # Enforce rate limit pacing
            elapsed_time = time.time() - last_request_time
            if elapsed_time < REQUEST_INTERVAL:
                time.sleep(REQUEST_INTERVAL - elapsed_time)

            # Make the API request
            
            completion = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=10,
                temperature=0,
            )

            # Update last request time
            last_request_time = time.time()

            # Extract and validate the classification
            classification = completion.choices[0].message.content.strip()

            if classification in {'0', '1', '2'}:
                return int(classification)  # Convert to integer
            
            print(f"Unexpected classification for tweet: {tweet}. Received: {classification}")
            return None  # Return None for unexpected responses

        except openai.APIError as e:
            retries += 1
            wait_time = 2 ** retries  # Exponential backoff (2, 4, 8, 16s)
            print(f"API error: {e}. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)

        except openai.RateLimitError:
            wait_time = 60  # Wait for a minute before retrying
            print(f"Rate limit exceeded. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)

    print(f"Failed to classify tweet after {max_retries} attempts.")
    return None  # Return None if all retries fail


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


def classify_tweets_parallel(df, text_column='text', num_workers=4):
    """Classifies tweets in parallel using ThreadPoolExecutor with rate limit handling."""
    results = []

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        # Use tqdm to show progress bar
        future_to_tweet = {executor.submit(classify_tweet, tweet): tweet for tweet in df[text_column]}

        for future in tqdm(as_completed(future_to_tweet), total=len(df), desc="Classifying tweets"):
            results.append(future.result())

    # Store results in dataframe
    df['gpt-classification'] = results
    return df

In [ ]:
# df_gpt_4o = classify_tweets_parallel(tweets[:10], text_column='text', num_workers=3)


In [24]:
import openai
import time
import re

REQUEST_INTERVAL = 60 / 60  # Adjust based on actual limits
MAX_RETRIES = 5  # Number of retry attempts

last_request_time = 0  # Initialize last request time

def classify_tweets_batch(tweets, max_retries=5, batch_size=5):
    """Classifies a batch of tweets using OpenAI, handling rate limits and retries."""
    global last_request_time

    # Split the tweets into batches
    tweet_batches = [tweets[i:i + batch_size] for i in range(0, len(tweets), batch_size)]
    
    all_classifications = []
    
    for batch in tweet_batches:
        # Construct the prompt for the batch
        prompt = "\n\n".join([f"""
        Please classify the following tweet based on its stance toward immigration, integration, refugees, or asylum.
        Be cautious about both direct and indirect quotes, and ensure that the speaker's stance is correctly identified.
        Use the number 1 for a positive stance,
        Use the number 2 for a negative stance,
        Use the number 0 for a neutral stance.

        Only return the numbers, never return any text or description.

        Tweet: {tweet}
        Classification:
        """ for tweet in batch])

        retries = 0
        while retries < max_retries:
            try:
                # Enforce rate limit pacing
                elapsed_time = time.time() - last_request_time
                if elapsed_time < REQUEST_INTERVAL:
                    time.sleep(REQUEST_INTERVAL - elapsed_time)

                # Make the API request
                completion = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=10,
                    temperature=0,
                )

                # Update last request time
                last_request_time = time.time()

                # Extract and validate the classifications
                classifications = [choice.message.content.strip() for choice in completion.choices]
                
                # Validate that all classifications are correct
                if all(classification in {'0', '1', '2'} for classification in classifications):
                    all_classifications.extend([int(classification) for classification in classifications])
                    break  # Exit loop if all classifications are valid

                print(f"Unexpected classification for batch: {batch}. Received: {classifications}")
                return None  # Return None for unexpected responses if any

            except openai.APIError as e:
                retries += 1
                wait_time = 2 ** retries  # Exponential backoff (2, 4, 8, 16s)
                print(f"API error: {e}. Retrying in {wait_time} seconds...")
                time.sleep(wait_time)

            except openai.RateLimitError as rate_error:
                # Check if the 'retry_after' field exists in the response
                retry_after = rate_error.response.json().get('error', {}).get('message', '')
                if 'Please try again in' in retry_after:
                    # Extract retry time from the error message
                    try:
                        retry_time = float(retry_after.split('in')[-1].strip().split()[0])
                    except ValueError:
                        retry_time = 60  # Default retry time if no valid number found
                else:
                    retry_time = 60  # Default retry time
                print(f"Rate limit exceeded. Retrying in {retry_time} seconds...")
                time.sleep(retry_time)

        if retries == max_retries:
            print(f"Failed to classify batch after {max_retries} attempts.")
            return None  # Return None if all retries fail for a batch

    return all_classifications  # Return the list of classifications for all tweets


In [25]:
classifications = classify_tweets_batch(tweets[:10], batch_size=5)


In [26]:
classifications

[0, 0]